In [ ]:
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm

import os
import sys
sys.path.append('../src')
import helper

In [ ]:
path = 'HelmLiteHeatmap'
#alternatives:
    #'HelmLiteHeatmap_0.9_0.1'
    #'HelmLiteHeatmap_0.1_0.4'

In [ ]:
folders = glob(f'../results/{path}/*')
folders = [p for p in folders if os.path.isdir(p)]

res=[]
for folder in tqdm(folders):
    models = glob(f'{folder}/*')
    models = [f for f in models if not f.endswith('runs')]
    for model in models:
        logits = np.load(f'{model}/Shuffle_logits.npy')
        labels = np.load(f'{model}/Shuffle_labels.npy')

        m = helper.compute_eval_metrics(logits, labels)
        source = model.split('/')[3]
        target = model.split('/')[4]

        res.append([source, target, m])

rows = []
for source, target, metrics in res:
    row = {'source': source, 'target': target}
    row.update(metrics)         
    rows.append(row)

df = pd.DataFrame(rows)

df.to_pickle(f'../results/{path}/heatmap_data.pkl')

In [ ]:
#compressed version
df = df.drop(columns=["aupr", "brier", "accuracy_0.5", "coverage_curve", "accuracy_curve"])
df.to_pickle(f'../results/{path}/heatmap_data_compressed.pkl')